# 01 – Data Wrangling & Base EDA for HDB Resale Prices

**Module**: ST1516 – DevOps & Applied Analytics  
**Project**: HDB Resale Price Prediction Web Application  
**Branch**: `data-wrangling`  
**Author**: \<Your Name\>  
**Date**: \<YYYY-MM-DD\>

---

## Purpose of This Notebook

This notebook performs the **foundational data engineering work** for the project.  
It takes the raw HDB resale transactions from `data/raw/hdb_resale_raw.csv` and:

1. Inspects and documents the structure and quality of the raw dataset.  
2. Performs **systematic cleaning**:
   - Type conversions (e.g. `month` to datetime, numeric parsing).
   - Handling of missing values for essential fields.
   - Removal of clearly invalid or extreme outliers.
3. Engineers **core non-geospatial features**:
   - `storey_mid` – numeric midpoint of the storey range.
   - `remaining_lease_years` – approximate remaining lease in years.
4. Conducts **baseline EDA** on:
   - Resale prices
   - Floor area
   - Remaining lease
   - Flat type and town
5. Exports a modelling-ready, but **pre-geocoding**, dataset as:

```text
data/intermediate/hdb_clean.csv


---

## 🔹 Section 1 – Imports & Display Settings



```
## 1. Environment Setup

We start by importing the core Python libraries used throughout this notebook and configuring display settings for easier inspection of wide tables.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

sns.set(style="whitegrid")

print("Libraries imported and display settings configured.")


Libraries imported and display settings configured.


## 2. Load Raw HDB Resale Dataset

We load the raw HDB resale transactions from:

```text
data/raw/hdb_resale_raw.csv


In [4]:
raw_path = "../data/raw/sg-resale-flat-prices-2017-onwards.csv"
df_raw = pd.read_csv(raw_path)

print(f"Loaded raw dataset from: {raw_path}")
print(f"Shape: {df_raw.shape}")
df_raw.head()

Loaded raw dataset from: ../data/raw/sg-resale-flat-prices-2017-onwards.csv
Shape: (181262, 11)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [6]:
df_raw.head()


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [7]:
df_raw.info()
df_raw.describe(include="all")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 181262 entries, 0 to 181261
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                181262 non-null  object 
 1   town                 181262 non-null  object 
 2   flat_type            181262 non-null  object 
 3   block                181262 non-null  object 
 4   street_name          181262 non-null  object 
 5   storey_range         181262 non-null  object 
 6   floor_area_sqm       181262 non-null  float64
 7   flat_model           181262 non-null  object 
 8   lease_commence_date  181262 non-null  int64  
 9   remaining_lease      181262 non-null  object 
 10  resale_price         181262 non-null  float64
dtypes: float64(2), int64(1), object(8)
memory usage: 15.2+ MB


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
count,181262,181262,181262,181262,181262,181262,181262.000000,181262,181262.000000,181262,1.812620e+05
unique,90,26,7,2708,566,17,NaN,21,NaN,676,NaN
top,2021-08,SENGKANG,4 ROOM,2,YISHUN RING RD,04 TO 06,NaN,Model A,NaN,94 years 10 months,NaN
freq,2739,14993,76457,554,2641,41701,NaN,63164,NaN,1643,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,97.152903,NaN,1996.013842,NaN,4.978643e+05
std,NaN,NaN,NaN,NaN,NaN,NaN,24.025926,NaN,14.041087,NaN,1.726320e+05
min,NaN,NaN,NaN,NaN,NaN,NaN,31.000000,NaN,1966.000000,NaN,1.400000e+05
25%,NaN,NaN,NaN,NaN,NaN,NaN,82.000000,NaN,1985.000000,NaN,3.700000e+05
50%,NaN,NaN,NaN,NaN,NaN,NaN,93.000000,NaN,1996.000000,NaN,4.680000e+05
75%,NaN,NaN,NaN,NaN,NaN,NaN,112.000000,NaN,2010.000000,NaN,5.920000e+05


## Raw Data Summary

The raw HDB resale dataset consists of **181,262 transactions across 11 variables**, with a total memory footprint of approximately **15.2 MB**. All columns exhibit **100% non-null completeness**, indicating that the dataset is structurally intact at the point of ingestion, with no missing records at source.

### Numeric Variables

- **Floor Area (`floor_area_sqm`)**
  - Mean: **97.15 sqm**, Median: **93 sqm**
  - Interquartile Range (IQR): **82–112 sqm**
  - Min–Max: **31–249 sqm**
  - This reflects a typical concentration around standard 4- and 5-room flats, with a small number of very large units (e.g., EXECUTIVE flats).

- **Lease Commencement Year (`lease_commence_date`)**
  - Median: **1996**
  - Range: **1966–2020**
  - The dataset spans over five decades of housing development, implying strong variation in remaining lease profiles across estates.

- **Resale Price (`resale_price`)**
  - Median: **SGD 468,000**
  - IQR: **SGD 370,000 – SGD 592,000**
  - Min–Max: **SGD 140,000 – SGD 1,588,000**
  - The extended upper tail indicates the presence of premium flats in prime locations or large configurations.

### Categorical Coverage

- **Towns**: 26 unique  
- **Flat Types**: 7 categories  
- **Flat Models**: 21 categories  
- **Blocks**: 2,708 unique blocks  
- **Street Names**: 566 unique streets  
- **Storey Ranges**: 17 bands  
- **Remaining Lease Formats**: 676 unique textual formats  

This confirms **high spatial and structural diversity**, but also signals significant **encoding complexity** for categorical modelling.
